In [173]:
# Imports
import numpy as np
import numpy.typing as npt
from sklearn.model_selection import train_test_split
import math
import tensorflow as tf
from sklearn.metrics import classification_report

I0000 00:00:1779821277.704553    1134 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779821278.775797    1134 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779821283.857649    1134 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [174]:
# Splitting data into train / cross-validation / test sets
X = np.load("data/players_batch_6_2026-05-25 21:44:14.801126.npy")
Y = np.load("data/results_batch_6_2026-05-25 21:44:14.801174.npy")
X_train, X_mid, Y_train, Y_mid = train_test_split(X, Y, test_size=0.2, random_state=42)
X_cv, X_test, Y_cv, Y_test = train_test_split(X_mid, Y_mid, test_size=0.5, random_state=42)
print(X_train.shape, X_cv.shape, X_test.shape)
print(Y_train.shape, Y_cv.shape, Y_test.shape)

(560, 22, 9) (70, 22, 9) (70, 22, 9)
(560, 121, 1) (70, 121, 1) (70, 121, 1)


In [183]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(22,9)),
    tf.keras.layers.Flatten(),
    # Predicting player quality
    tf.keras.layers.Dense(22, activation='relu'),
    
    # Predicting team that will win
    tf.keras.layers.Dense(121, activation='linear')
])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 198)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 22)             │         4,378 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 121)            │         2,783 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,161 (27.97 KB)

 Trainable params: 7,161 (27.97 KB)

 Non-trainable params: 0 (0.00 B)

In [184]:
model.compile(
    loss = tf.keras.losses.CategoricalCrossentropy(from_logits=True)
)

In [185]:
model.fit(
    X_train,
    Y_train,
    epochs=100
)

Epoch 1/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 49.7959    
Epoch 2/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 18.3391 
Epoch 3/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 9.4774  
Epoch 4/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 6.0951 
Epoch 5/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 4.9793 
Epoch 6/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 4.4861 
Epoch 7/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3.9974 
Epoch 8/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3.3672 
Epoch 9/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2.7346 
Epoch 10/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2.2222 
Epoch 11/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2.0671 
Epoch 12/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2.0178 
Epoch 13/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.9743 
Epoch 14/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.9479 
Epoch 15/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

In [186]:
model.evaluate(
    X_cv,
    Y_cv
)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 4.0744 


4.074411869049072

In [187]:
def interpret_label(label: int):
    """ 
    Print text version of a label
    """
    home_score = math.floor(label / 11)
    away_score = label % 11
    return f"{home_score}-{away_score}"

In [188]:
labels = list(range(0,122))
labels_text = [interpret_label(l) for l in labels]
print(labels_text)
logits = model.predict(X_cv)
y_pred = np.argmax(logits, axis=1)
y_cv = np.argmax(Y_cv, axis=1)
print(classification_report(y_true=y_cv, y_pred=y_pred, labels=labels, target_names=labels_text, zero_division=0))

['0-0', '0-1', '0-2', '0-3', '0-4', '0-5', '0-6', '0-7', '0-8', '0-9', '0-10', '1-0', '1-1', '1-2', '1-3', '1-4', '1-5', '1-6', '1-7', '1-8', '1-9', '1-10', '2-0', '2-1', '2-2', '2-3', '2-4', '2-5', '2-6', '2-7', '2-8', '2-9', '2-10', '3-0', '3-1', '3-2', '3-3', '3-4', '3-5', '3-6', '3-7', '3-8', '3-9', '3-10', '4-0', '4-1', '4-2', '4-3', '4-4', '4-5', '4-6', '4-7', '4-8', '4-9', '4-10', '5-0', '5-1', '5-2', '5-3', '5-4', '5-5', '5-6', '5-7', '5-8', '5-9', '5-10', '6-0', '6-1', '6-2', '6-3', '6-4', '6-5', '6-6', '6-7', '6-8', '6-9', '6-10', '7-0', '7-1', '7-2', '7-3', '7-4', '7-5', '7-6', '7-7', '7-8', '7-9', '7-10', '8-0', '8-1', '8-2', '8-3', '8-4', '8-5', '8-6', '8-7', '8-8', '8-9', '8-10', '9-0', '9-1', '9-2', '9-3', '9-4', '9-5', '9-6', '9-7', '9-8', '9-9', '9-10', '10-0', '10-1', '10-2', '10-3', '10-4', '10-5', '10-6', '10-7', '10-8', '10-9', '10-10', '11-0']
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
              precision    recall  f1-score   support

         0-0       0.00      